# HAI Digital Twin — Transformer Training on Google Colab GPU

In [19]:
# 1. Mount Google Drive (your data must be in Drive)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# 2. Clone the repo
!git clone https://github.com/hayooka/hai-digital-twin.git
%cd hai-digital-twin

Cloning into 'hai-digital-twin'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 104 (delta 43), reused 55 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 457.68 KiB | 3.16 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/hai-digital-twin/hai-digital-twin/hai-digital-twin


In [21]:
# 3. Install dependencies
!pip install -q torch numpy pandas scikit-learn

In [22]:
# 4. Set data paths — EDIT THESE to match your Google Drive folder
import os

# Change these paths to where your data is in Google Drive
os.environ['HAI_DIR']   = '/content/drive/MyDrive/hai-23.05'
os.environ['HIEND_DIR'] = '/content/drive/MyDrive/haiend-23.05'

# Verify files exist
print('HAI files:',   os.listdir(os.environ['HAI_DIR'])[:5])
print('HIEND files:', os.listdir(os.environ['HIEND_DIR'])[:5])

HAI files: ['hai-test2.csv', 'hai-test1.csv', 'hai-train1.csv', 'summary_label2.txt', 'label-test2.csv']
HIEND files: ['end-test1.csv', 'end-test2.csv', 'end-train1.csv', 'end-train2.csv', 'end-train3.csv']


In [23]:
# 5. Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

GPU available: True
Device: Tesla T4


In [24]:
# Patch for Colab RAM limits
import re

with open('models/transformer_model.py', 'r') as f:
    code = f.read()

code = code.replace('stride=60', 'stride=180')
code = code.replace('D_MODEL  = 256', 'D_MODEL  = 128')
code = code.replace('N_LAYERS = 4', 'N_LAYERS = 3')
code = code.replace('FFN_DIM  = 1024', 'FFN_DIM  = 512')

with open('models/transformer_model.py', 'w') as f:
    f.write(code)

print("Patched for Colab!")


Patched for Colab!


In [ ]:
# 6. Run training!
!python models/transformer_model.py

[train1] merged: 280799 rows, 279 cols (dropped 34 duplicate HAI cols)
[train2] merged: 291599 rows, 279 cols (dropped 34 duplicate HAI cols)
[train3] merged: 125999 rows, 279 cols (dropped 34 duplicate HAI cols)
[train4] merged: 197999 rows, 279 cols (dropped 34 duplicate HAI cols)
  └─ loaded labels from label-test1.csv  (attacks: 2981)
[test1] merged: 53999 rows, 279 cols (dropped 34 duplicate HAI cols)
  └─ loaded labels from label-test2.csv  (attacks: 130)
[test2] merged: 230399 rows, 279 cols (dropped 34 duplicate HAI cols)
Digital Twin data ready:
  X_train (3877, 60, 277)  Y_train (3877, 180, 277)
  X_val   (1099, 60, 277)    Y_val   (1099, 180, 277)
  X_test  (1578, 60, 277)   Y_test  (1578, 180, 277)
Parameters: 1,459,861
Epoch   1/50  train=0.64176  val=0.40290
Epoch  10/50  train=0.07335  val=0.06631
Epoch  20/50  train=0.04486  val=0.04096
Epoch  30/50  train=0.03498  val=0.03112
Epoch  40/50  train=0.02966  val=0.02598
Epoch  50/50  train=0.02669  val=0.02315
Best val los

In [ ]:
# 7. Download the trained model
from google.colab import files
files.download('outputs/transformer_twin.pt')
files.download('outputs/transformer_metrics.json')